# Optiver realized volatility (late submission)

This is the best model from our notebook 05 (Erdős Institute project, Fall 2026), put into one notebook so it runs on Kaggle. It uses features from the whole ten-minute window to predict RV for the next ten minutes.

The model is 80% neural net (averaged over 3 seeds) + 20% LightGBM, both trained on RMSPE. CV 0.1954.

`USE_NEIGHBOURS` switches the neighbour features on or off (the trick the top solutions used, averaging features over the most similar windows, including the 1–3 closest by recovered price). Neighbours are searched over train and test windows together.

Earlier submissions (single XGBoost): 0.2455 private without neighbours, 0.2186 with the first neighbour version, 0.2141 with the closest 1–3 windows added.

In [ ]:
USE_NEIGHBOURS = True
W_NN = 0.8                  # weight on the neural net, rest goes to LightGBM
N_SEEDS = 3                 # neural nets averaged over this many random seeds
NN_EPOCHS = 30
N_JOBS = 4                  # Kaggle notebooks have 4 CPUs (GPU is used if switched on)

Feature code (copied from `src/obforecast/volatility.py` in our repo).

In [ ]:
"""Realized-volatility features in the style of the Kaggle competition notebooks.

The Optiver competition asked for the realized volatility (RV) of the weighted average price
(WAP) over the ten minutes after each window. Public notebooks converged on a common recipe:

* RV of WAP log returns over the whole window and over its later parts;
* book state (spreads, depth, imbalance) and trade activity summarised over the same parts;
* the same features averaged over every stock in the window ("market" features), because
  volatility moves together across stocks;
* models trained on RMSPE, the competition metric, through weights of 1/y^2.

The winning solutions added features from other windows that looked similar (nearest
neighbours by recovered price level or by the cross-section of volatilities). Because those
neighbours can lie later in calendar time, those features are look-ahead in a real-time setting;
``neighbour_features`` exists so that the gap can be measured, and the notebook reports it
separately.

Two framings are supported through ``cut``:

* ``cut=600``: the competition task. Features use the whole window; the target is the RV of the
  next ten minutes, given in ``train.csv``.
* ``cut=300``: this project's task. Features use seconds 0..299; the target is the RV of WAP1
  over seconds 300..599 of the same window (``rv_target``), computed from the raw snapshots.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

KEYS = ["stock_id", "time_id"]


def _group_index(time_id: np.ndarray):
    tids, idx = np.unique(time_id, return_inverse=True)
    return tids, idx


def _sum(idx, n, x, mask=None):
    x = np.where(np.isfinite(x), x, 0.0)
    if mask is not None:
        x = np.where(mask, x, 0.0)
    return np.bincount(idx, weights=x, minlength=n)


def _count(idx, n, mask):
    return np.bincount(idx, weights=mask.astype(float), minlength=n)


def _mean(idx, n, x, mask):
    ok = mask & np.isfinite(x)
    c = _count(idx, n, ok)
    with np.errstate(invalid="ignore", divide="ignore"):
        return _sum(idx, n, x, ok) / c


def _std(idx, n, x, mask):
    m = _mean(idx, n, x, mask)
    ok = mask & np.isfinite(x)
    c = _count(idx, n, ok)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.sqrt(_sum(idx, n, (x - m[idx]) ** 2, ok) / c)


def _log_returns(price: np.ndarray, time_id: np.ndarray) -> np.ndarray:
    with np.errstate(divide="ignore", invalid="ignore"):
        lp = np.log(price)
    r = np.diff(lp, prepend=np.nan)
    first = np.r_[True, time_id[1:] != time_id[:-1]]
    r[first] = np.nan
    return r


def book_frame(book: pd.DataFrame) -> pd.DataFrame:
    """Row-level derived quantities on the raw snapshots (no forward filling)."""
    b = book.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)
    bp1, ap1 = b.bid_price1.to_numpy(float), b.ask_price1.to_numpy(float)
    bp2, ap2 = b.bid_price2.to_numpy(float), b.ask_price2.to_numpy(float)
    bs1, as1 = b.bid_size1.to_numpy(float), b.ask_size1.to_numpy(float)
    bs2, as2 = b.bid_size2.to_numpy(float), b.ask_size2.to_numpy(float)
    tid = b.time_id.to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        wap1 = (bp1 * as1 + ap1 * bs1) / (bs1 + as1)
        wap2 = (bp2 * as2 + ap2 * bs2) / (bs2 + as2)
        mid = (ap1 + bp1) / 2
        out = pd.DataFrame({
            "time_id": tid, "sec": b.seconds_in_bucket.to_numpy(),
            "r1": _log_returns(wap1, tid), "r2": _log_returns(wap2, tid),
            "spread": (ap1 - bp1) / mid,
            "spread2": (ap2 - bp2) / mid,
            "bid_gap": (bp1 - bp2) / mid,
            "ask_gap": (ap2 - ap1) / mid,
            "wap_bal": np.abs(wap1 - wap2) / mid,
            "depth": bs1 + as1 + bs2 + as2,
            "imb": np.abs((bs1 + bs2) - (as1 + as2)) / (bs1 + as1 + bs2 + as2),
        })
    return out


def price_level(book: pd.DataFrame) -> pd.Series:
    """Recovered price level per window, the 1st-place trick.

    Prices in the data are normalised to start near 1, but they still move in steps of one cent
    of the real price. The smallest gap between distinct price values is therefore about
    0.01 / real_price, so 0.01 / gap estimates the real price. Windows of one stock with similar
    real prices are close in calendar time.
    """
    px = book[["time_id", "bid_price1", "ask_price1", "bid_price2", "ask_price2"]]
    long = px.melt(id_vars="time_id", value_name="p")[["time_id", "p"]].drop_duplicates()
    long = long.sort_values(["time_id", "p"])
    gap = long.groupby("time_id").p.diff()
    gap = gap[gap > 0]
    tick = gap.groupby(long.loc[gap.index, "time_id"]).min()
    return (0.01 / tick).rename("price_level")


def window_features(bf: pd.DataFrame, trades: pd.DataFrame | None, cut: int,
                    starts: list[int]) -> pd.DataFrame:
    """Per-window summaries of book rows (and trades) with ``start <= sec < cut``."""
    tids, idx = _group_index(bf.time_id.to_numpy())
    n = len(tids)
    sec = bf.sec.to_numpy()
    out = {"time_id": tids}
    for s in starts:
        m = (sec >= s) & (sec < cut)
        sfx = f"_{s}"
        for r in ("r1", "r2"):
            x = bf[r].to_numpy()
            ok = m & np.isfinite(x)
            out[f"rv{r[1]}{sfx}"] = np.sqrt(_sum(idx, n, x ** 2, ok))
        out[f"n_book{sfx}"] = _count(idx, n, m)
        if s == starts[0]:
            for c in ("spread", "spread2", "bid_gap", "ask_gap", "wap_bal", "depth", "imb"):
                out[f"{c}_mean"] = _mean(idx, n, bf[c].to_numpy(), m)
            out["spread_std"] = _std(idx, n, bf["spread"].to_numpy(), m)
            out["depth_std"] = _std(idx, n, bf["depth"].to_numpy(), m)
            r1 = bf["r1"].to_numpy()
            ok = m & np.isfinite(r1)
            q = _sum(idx, n, r1 ** 4, ok)
            rv2 = _sum(idx, n, r1 ** 2, ok)
            with np.errstate(invalid="ignore", divide="ignore"):
                out["rquart"] = np.log(np.sqrt(np.sqrt(q)) + 1e-8)    # realized quarticity, 4th root
                bp = np.abs(r1) * np.abs(np.r_[np.nan, r1[:-1]])         # bipower: jump-robust
                same = np.r_[False, bf.time_id.to_numpy()[1:] == bf.time_id.to_numpy()[:-1]]
                out["bipower_ratio"] = (np.pi / 2) * _sum(idx, n, bp, ok & same) / rv2
    f = pd.DataFrame(out)

    if trades is not None:
        t = trades[trades.seconds_in_bucket < cut].sort_values(["time_id", "seconds_in_bucket"])
        tt = t.time_id.to_numpy()
        tr = _log_returns(t.price.to_numpy(float), tt)
        tpos = np.searchsorted(tids, tt)
        valid = (tpos < n) & (tids[np.minimum(tpos, n - 1)] == tt)
        tpos, ts = tpos[valid], t.seconds_in_bucket.to_numpy()[valid]
        tr = tr[valid]
        size = t["size"].to_numpy(float)[valid]
        oc = t.order_count.to_numpy(float)[valid]
        for s in starts:
            m = ts >= s
            sfx = f"_{s}"
            f[f"trv{sfx}"] = np.sqrt(_sum(tpos, n, tr ** 2, m & np.isfinite(tr)))
            f[f"n_trade{sfx}"] = _count(tpos, n, m)
            f[f"size{sfx}"] = _sum(tpos, n, size, m)
            f[f"orders{sfx}"] = _sum(tpos, n, oc, m)
        with np.errstate(invalid="ignore", divide="ignore"):
            f["size_per_order"] = f[f"size_{starts[0]}"] / f[f"orders_{starts[0]}"]
    return f


def rv_target(bf: pd.DataFrame, cut: int) -> pd.DataFrame:
    """RV of WAP1 returns that end at or after second ``cut``, per window (``cut=300`` task)."""
    tids, idx = _group_index(bf.time_id.to_numpy())
    r = bf.r1.to_numpy()
    m = (bf.sec.to_numpy() >= cut) & np.isfinite(r)
    return pd.DataFrame({"time_id": tids, "target": np.sqrt(_sum(idx, len(tids), r ** 2, m)),
                         "n_after": _count(idx, len(tids), bf.sec.to_numpy() >= cut)})


def build_stock(stock_id: int, book: pd.DataFrame, trades: pd.DataFrame | None, cut: int,
                starts: list[int]) -> pd.DataFrame:
    bf = book_frame(book)
    f = window_features(bf, trades, cut, starts)
    f = f.merge(price_level(book[book.seconds_in_bucket < cut]).reset_index(), on="time_id", how="left")
    if cut < 600:
        f = f.merge(rv_target(bf, cut), on="time_id", how="left")
    f.insert(0, "stock_id", int(stock_id))
    return f


def add_log_features(df: pd.DataFrame) -> list[str]:
    """Log-transform the scale-like columns in place; return the model feature list."""
    skip = set(KEYS) | {"target", "n_after", "fold", "bipower_ratio", "rquart", "imb_mean",
                        "price_level"}
    feats = []
    for c in df.columns:
        if c in skip or c.startswith("y_"):
            continue
        if (df[c].dropna() >= 0).all():
            df[c] = np.log(df[c] + 1e-8)
        feats.append(c)
    return feats + ["bipower_ratio", "rquart", "imb_mean"]


def add_market_features(df: pd.DataFrame, cols: list[str]) -> list[str]:
    """Cross-sectional mean and std over stocks in the same window, plus each stock's deviation.

    Uses only features observed at the cut, so it is available in real time.
    """
    g = df.groupby("time_id")[cols]
    mean, std = g.transform("mean"), g.transform("std")
    new = []
    for c in cols:
        df[f"mkt_{c}"] = mean[c]
        df[f"mktsd_{c}"] = std[c]
        df[f"dev_{c}"] = df[c] - mean[c]
        new += [f"mkt_{c}", f"mktsd_{c}", f"dev_{c}"]
    return new


def add_stock_features(df: pd.DataFrame, cols: list[str], train_mask: np.ndarray) -> list[str]:
    """Each stock's average of ``cols`` over training windows only."""
    s = df[train_mask].groupby("stock_id")[cols].mean().add_prefix("stk_")
    for c in s.columns:
        df[c] = df.stock_id.map(s[c])
    return list(s.columns)


def _window_matrix(df: pd.DataFrame, pivot_col: str, metric: str):
    """One row per window: its cross-sectional vector of ``pivot_col``, scaled for distances."""
    piv = df.pivot(index="time_id", columns="stock_id", values=pivot_col)
    piv = piv.fillna(piv.mean())
    X = piv.to_numpy()
    if metric == "corr":
        X = (X - X.mean(0)) / (X.std(0) + 1e-12)
    else:                       # min-max per stock, as in the 1st-place price-level neighbours
        X = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-12)
    return piv.index.to_numpy(), X


def neighbour_features(df: pd.DataFrame, pivot_col: str, value_cols: list[str],
                       ks: list[int], name: str, metric: str = "corr") -> list[str]:
    """Average of ``value_cols`` over each window's k most similar other windows.

    Similarity is between windows' cross-sectional vectors of ``pivot_col`` (one value per
    stock). Uses features only, never targets, but neighbours may be later in calendar time.
    Small k (1-3) with price-level neighbours picks out the windows just before and after.
    """
    from sklearn.neighbors import NearestNeighbors

    tids, X = _window_matrix(df, pivot_col, metric)
    nn = NearestNeighbors(n_neighbors=max(ks) + 1).fit(X)
    _, ind = nn.kneighbors(X)
    ind = ind[:, 1:]                                  # drop the window itself
    new = []
    pos = pd.Series(np.arange(len(tids)), index=tids)
    row_t = pos.loc[df.time_id].to_numpy()
    for c in value_cols:
        vp = df.pivot(index="time_id", columns="stock_id", values=c).reindex(tids)
        V = vp.to_numpy()
        col_s = vp.columns.get_indexer(df.stock_id)
        for k in ks:
            nb = np.nanmean(V[ind[:, :k]], axis=1)    # (window, stock)
            df[f"nn{name}{k}_{c}"] = nb[row_t, col_s]
            new.append(f"nn{name}{k}_{c}")
    return new


def rmspe(y, p) -> float:
    y, p = np.asarray(y), np.asarray(p)
    return float(np.sqrt(np.mean(((y - p) / y) ** 2)))


In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)
def find_data():
    # The competition folder's location under /kaggle/input varies, so look for it.
    if os.environ.get("OBF_DATA_DIR"):
        return Path(os.environ["OBF_DATA_DIR"])
    root = Path("/kaggle/input")
    for csv in sorted(root.rglob("train.csv")):
        if (csv.parent / "book_train.parquet").is_dir():
            return csv.parent
    mounted = "\n  ".join(str(p) for p in sorted(root.glob("*/*"))[:50]) or "(nothing)"
    raise FileNotFoundError("Competition data not found under /kaggle/input. Add it with 'Add Input' -> "
                            f"Competitions -> Optiver Realized Volatility Prediction. Mounted now:\n  {mounted}")


DATA = find_data()
print("data:", DATA)
STARTS = [0, 150, 300, 450]
TRADE_COLS = ["time_id", "seconds_in_bucket", "price", "size", "order_count"]


def load(kind, split, sid):
    paths = sorted((DATA / f"{kind}_{split}.parquet" / f"stock_id={sid}").glob("*.parquet"))
    if not paths:
        return None
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


def one(split, sid):
    book = load("book", split, sid)
    if book is None:
        return None
    trades = load("trade", split, sid)
    if trades is None:
        trades = pd.DataFrame({c: pd.Series(dtype=float) for c in TRADE_COLS})
    return build_stock(sid, book, trades, 600, STARTS)


train_y = pd.read_csv(DATA / "train.csv")
test_ids = pd.read_csv(DATA / "test.csv")
parts = {}
for split, ids in [("train", train_y), ("test", test_ids)]:
    out = Parallel(n_jobs=N_JOBS)(delayed(one)(split, s) for s in sorted(ids.stock_id.unique()))
    parts[split] = pd.concat([o for o in out if o is not None], ignore_index=True)

train = parts["train"].merge(train_y, on=KEYS, how="inner")
test = test_ids[KEYS].merge(parts["test"], on=KEYS, how="left")   # keep every requested row
test["time_id"] = test.time_id + 1_000_000                      # keep test windows distinct from train
df = pd.concat([train, test], ignore_index=True)
is_test = np.r_[np.zeros(len(train), bool), np.ones(len(test), bool)]
print(f"train rows {len(train):,}, test rows {len(test):,}")

In [ ]:
base = add_log_features(df)
is_trade = lambda c: c.startswith(("trv", "n_trade", "size", "orders"))
cols = [c for c in base if c != "price_level"]
cols += add_market_features(df, ["rv1_0", "rv1_300", "rv1_450", "rv2_0", "trv_0", "n_trade_0",
                                 "spread_mean", "depth_mean", "n_book_0", "size_0"])
if USE_NEIGHBOURS:
    # k = 1-3 by recovered price picks out the windows just before and after each window
    nn_src = ["rv1_0", "rv1_150", "rv1_300", "rv1_450", "rv2_0", "trv_0", "n_trade_0", "spread_mean", "depth_mean"]
    cols += neighbour_features(df, "rv1_0", nn_src, [3, 10, 50], "vol", metric="corr")
    df["log_price_level"] = np.log(df.price_level)
    cols += neighbour_features(df, "log_price_level", nn_src, [1, 2, 3, 5, 10, 20, 50], "px", metric="minmax")

# stock averages from training windows only
cols += add_stock_features(df, ["rv1_0", "trv_0", "n_trade_0"], ~is_test)
y = df.target.to_numpy()
df["stk_log_target"] = df.stock_id.map(pd.Series(np.log(y[~is_test])).groupby(df.stock_id[~is_test].to_numpy()).mean())
cols.append("stk_log_target")
print(f"{len(cols)} model inputs")

Two models: LightGBM and a small neural net, both trained on RMSPE. The final forecast is 80% neural net + 20% LightGBM (weights picked on our CV: 0.1954 vs 0.1958 for the neural net alone).

In [ ]:
import lightgbm as lgb
import torch
from torch import nn
from sklearn.preprocessing import QuantileTransformer

Xtr, Xte, ytr = df.loc[~is_test, cols], df.loc[is_test, cols], y[~is_test]
w = 1 / ytr ** 2
w = w / w.mean()

# LightGBM
lgbm = lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.03, num_leaves=127, min_child_samples=300,
                         subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                         n_jobs=N_JOBS, random_state=0, verbose=-1)
pred_lgb = lgbm.fit(Xtr, ytr, sample_weight=w).predict(Xte)
print("lightgbm done")

In [ ]:
# Neural net: quantile-scaled inputs + a learned vector per stock, loss = RMSPE
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(N_JOBS)
A_np, B_np = Xtr.to_numpy(np.float32), Xte.to_numpy(np.float32)
med = np.nan_to_num(np.nanmedian(A_np, axis=0))
fill = lambda X: np.where(np.isfinite(X), X, med)
qt = QuantileTransformer(n_quantiles=2000, output_distribution="normal", subsample=200_000, random_state=0)
A = torch.tensor(qt.fit_transform(fill(A_np)), dtype=torch.float32)
B = torch.tensor(qt.transform(fill(B_np)), dtype=torch.float32)
yt = torch.tensor(ytr, dtype=torch.float32)
st = torch.tensor(df.stock_id.to_numpy()[~is_test], dtype=torch.long)
sv = torch.tensor(df.stock_id.to_numpy()[is_test], dtype=torch.long)
scale = float(np.exp(np.mean(np.log(ytr))))


class Net(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.emb = nn.Embedding(128, 16)
        self.body = nn.Sequential(nn.Linear(d + 16, 384), nn.SiLU(), nn.Dropout(0.1),
                                  nn.Linear(384, 192), nn.SiLU(), nn.Dropout(0.1),
                                  nn.Linear(192, 96), nn.SiLU(), nn.Linear(96, 1))

    def forward(self, x, s):
        return scale * torch.exp(self.body(torch.cat([x, self.emb(s)], 1)).squeeze(1))


def train_nn(seed, epochs=NN_EPOCHS, bs=2048):
    torch.manual_seed(seed)
    net = Net(A.shape[1]).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=epochs * (len(A) // bs + 1))
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(len(A))
        for i in range(0, len(A), bs):
            idx = perm[i:i + bs]
            p, yy = net(A[idx].to(dev), st[idx].to(dev)), yt[idx].to(dev)
            loss = torch.sqrt(torch.mean(((yy - p) / yy) ** 2))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    net.eval()
    with torch.no_grad():
        return np.concatenate([net(B[i:i + 16384].to(dev), sv[i:i + 16384].to(dev)).cpu().numpy()
                               for i in range(0, len(B), 16384)])


pred_nn = np.mean([train_nn(seed) for seed in range(N_SEEDS)], axis=0)
print("neural net done")

In [ ]:
pred = W_NN * pred_nn + (1 - W_NN) * pred_lgb
fallback = np.exp(df.loc[is_test, "rv1_0"].to_numpy()) - 1e-8     # last-window RV if a prediction is unusable
pred = np.where(np.isfinite(pred) & (pred > 0), pred, fallback)
pred = np.where(np.isfinite(pred) & (pred > 0), pred, np.nanmedian(ytr))
sub = pd.DataFrame({"row_id": test_ids.stock_id.astype(str) + "-" + test_ids.time_id.astype(str), "target": pred})
sub.to_csv("submission.csv", index=False)
sub.head()